# 📖 Module 07: RAG Chain — End to End

## GenAI L2 Exam Preparation

**Topics Covered:**
- Full RAG pipeline: Load → Chunk → Embed → Store → Retrieve → Generate
- RetrievalQA chain
- LCEL (LangChain Expression Language) chains
- Chain types: stuff, map_reduce, refine, map_rerank
- Source document tracking

**Source Material:** Class 31 (RAG Pipeline End-to-End)

---

## 1. Complete RAG Pipeline

This module puts all previous modules together into a working end-to-end system.

```
Step 1: LOAD      → Document Loaders (Module 02)
Step 2: CHUNK     → Text Splitters (Module 03)
Step 3: EMBED     → Embedding Models (Module 04)
Step 4: STORE     → Vector Databases (Module 05)
Step 5: RETRIEVE  → Retrievers (Module 06)
Step 6: GENERATE  → LLM + Chain (This Module!) ⬅️
```

In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()

print("✅ Environment loaded")

## 2. Full Pipeline — Step by Step

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 1: LOAD — Load documents
# ═══════════════════════════════════════════════════════════
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./data/sample.txt", encoding="utf-8")
documents = loader.load()

print(f"📄 Step 1 — LOAD: {len(documents)} document(s) loaded")
print(f"   Total characters: {sum(len(d.page_content) for d in documents)}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 2: CHUNK — Split into smaller pieces
# ═══════════════════════════════════════════════════════════
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"📝 Step 2 — CHUNK: {len(documents)} doc → {len(chunks)} chunks")
print(f"   chunk_size=300, chunk_overlap=50")
for i, chunk in enumerate(chunks[:3]):
    print(f"   Chunk {i+1}: {len(chunk.page_content)} chars")

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 3 & 4: EMBED + STORE — Create embeddings and vector DB
# ═══════════════════════════════════════════════════════════
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"🔢 Step 3 — EMBED: Using all-MiniLM-L6-v2 ({len(embeddings.embed_query('test'))}d)")
print(f"🗄️ Step 4 — STORE: {len(chunks)} vectors stored in FAISS")

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 5: RETRIEVE — Create retriever
# ═══════════════════════════════════════════════════════════
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Test retrieval
test_query = "What is RAG?"
test_results = retriever.invoke(test_query)

print(f"🔍 Step 5 — RETRIEVE: Top {len(test_results)} results for '{test_query}'")
for i, doc in enumerate(test_results, 1):
    print(f"   [{i}] {doc.page_content[:80]}...")

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 6: GENERATE — Build chain and generate answers
# ═══════════════════════════════════════════════════════════
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize LLM
llm = ChatGroq(model="llama-3.1-8b-instant")

# RAG Prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that answers questions based on the provided context.
Rules:
- Only use information from the context below
- If the context doesn't contain the answer, say "I don't have enough information"
- Be concise and accurate"""),
    ("human", """Context:
{context}

Question: {question}

Answer:""")
])

# Helper: Format retrieved documents into a context string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build LCEL Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✅ Step 6 — RAG Chain built!")
print("🎯 Pipeline: Retriever → Format → Prompt → LLM → Parse")

In [ ]:
# Test the complete pipeline!
questions = [
    "What is RAG?",
    "What are Large Language Models?",
    "What is the difference between machine learning and deep learning?",
]

try:
    for q in questions:
        print(f"❓ {q}")
        answer = rag_chain.invoke(q)
        print(f"🤖 {answer}")
        print("-" * 60)
except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Ensure GROQ_API_KEY is set in .env")

## 3. RetrievalQA Chain (Legacy but still tested!)

The `RetrievalQA` chain is the older, simpler way to build RAG.

In [ ]:
from langchain.chains import RetrievalQA

try:
    qa_chain = RetrievalQA.from_chain_type(
        llm=ChatGroq(model="llama-3.1-8b-instant"),
        chain_type="stuff",            # How to handle multiple docs
        retriever=retriever,
        return_source_documents=True    # Include source docs in output
    )
    
    result = qa_chain.invoke({"query": "What is Natural Language Processing?"})
    
    print(f"❓ Query: {result['query']}")
    print(f"🤖 Answer: {result['result']}")
    print(f"\n📚 Sources:")
    for i, doc in enumerate(result['source_documents'], 1):
        print(f"  [{i}] {doc.page_content[:80]}...")

except Exception as e:
    print(f"⚠️ Error: {e}")

## 4. ⭐ Chain Types (EXAM CRITICAL!)

The `chain_type` parameter in `RetrievalQA` determines how retrieved documents are processed:

| Chain Type | How it works | Pros | Cons | Use When |
|-----------|-------------|------|------|----------|
| **stuff** | All docs stuffed into one prompt | Simple, fast, single LLM call | Limited by context window | Few/small documents ⭐ default |
| **map_reduce** | Each doc processed separately, then combined | Handles many large docs | Multiple LLM calls (slow, costly) | Many documents |
| **refine** | Iteratively refines answer with each doc | Good quality, considers order | Sequential (slowest) | Incremental detail needed |
| **map_rerank** | Each doc gets scored, highest score wins | Selects best single answer | May miss combined context | Need best single answer |

### Visual Comparison

```
STUFF:       [Doc1 + Doc2 + Doc3] → LLM → Answer

MAP_REDUCE:  Doc1 → LLM → Summary1 ─┐
             Doc2 → LLM → Summary2 ─┤→ LLM → Final Answer
             Doc3 → LLM → Summary3 ─┘

REFINE:      Doc1 → LLM → Answer_v1
             Answer_v1 + Doc2 → LLM → Answer_v2
             Answer_v2 + Doc3 → LLM → Answer_v3 (final)

MAP_RERANK:  Doc1 → LLM → (Answer1, Score: 0.3)
             Doc2 → LLM → (Answer2, Score: 0.9) ← Winner!
             Doc3 → LLM → (Answer3, Score: 0.5)
```

### 🎯 Exam Tip
> Default is **stuff** — works for most cases when context fits in the window.  
> If the exam asks "too many documents for the context window" → **map_reduce**  
> If it asks "need to refine answer with each additional document" → **refine**

## 5. LCEL vs Legacy Chains

| Feature | LCEL (Modern) | RetrievalQA (Legacy) |
|---------|---------------|---------------------|
| **Syntax** | `chain = prompt \| llm \| parser` | `RetrievalQA.from_chain_type()` |
| **Flexibility** | Highly customizable | Pre-built, less flexible |
| **Streaming** | Built-in support | Limited |
| **Async** | Native async support | Limited |
| **Recommended** | ✅ Yes (modern approach) | Still works but being deprecated |

### LCEL Pipe Operator (`|`)
```python
# Each component's output becomes the next component's input
chain = prompt | llm | output_parser
#        ↓        ↓        ↓
#     Format → Generate → Parse
```

### 🎯 Exam Tip
> Know **both** styles — LCEL is the modern way, but RetrievalQA is still in exams.  
> LCEL uses the `|` (pipe) operator, similar to Unix pipes.

## 6. Adding Source Attribution

In [ ]:
# LCEL chain with source tracking
from langchain_core.runnables import RunnableParallel

try:
    # Chain that returns both answer and sources
    rag_chain_with_sources = RunnableParallel(
        {
            "answer": (
                {"context": retriever | format_docs, "question": RunnablePassthrough()}
                | rag_prompt
                | llm
                | StrOutputParser()
            ),
            "source_documents": retriever
        }
    )

    result = rag_chain_with_sources.invoke("What is deep learning?")

    print(f"🤖 Answer: {result['answer']}")
    print(f"\n📚 Sources:")
    for i, doc in enumerate(result['source_documents'], 1):
        print(f"  [{i}] {doc.metadata.get('source', 'unknown')}")

except Exception as e:
    print(f"⚠️ Error: {e}")

## 🧠 Self-Assessment Quiz

---

**Q1.** What are the 6 stages of a RAG pipeline in order?

<details>
<summary>Click for Answer</summary>

1. **Load** — Load documents (DocumentLoader)  
2. **Chunk** — Split into pieces (TextSplitter)  
3. **Embed** — Convert to vectors (Embeddings)  
4. **Store** — Save in vector DB (VectorStore)  
5. **Retrieve** — Find relevant docs (Retriever)  
6. **Generate** — LLM produces answer (Chain)
</details>

---

**Q2.** You have 50 long documents and they exceed the LLM's context window when stuffed together. Which chain_type should you use?

<details>
<summary>Click for Answer</summary>

**map_reduce** — it processes each document separately with the LLM, then combines the summaries. This avoids context window overflow. Trade-off: multiple LLM calls = slower and more expensive.
</details>

---

**Q3.** What does `RunnablePassthrough()` do in an LCEL chain?

<details>
<summary>Click for Answer</summary>

`RunnablePassthrough()` **passes the input through unchanged**. In a RAG chain, it's used to forward the user's question directly to the prompt template while the retriever processes it separately for context retrieval.
</details>

---

**Q4.** What is the difference between `stuff` and `refine` chain types?

<details>
<summary>Click for Answer</summary>

- **stuff**: All documents are concatenated and sent to the LLM in a **single call**  
- **refine**: Documents are processed **one at a time**, with each step refining the previous answer  
Stuff is faster (1 call); refine produces higher quality but is slowest (N calls, sequential).
</details>

---

**Q5.** How do you get source documents with `RetrievalQA`?

<details>
<summary>Click for Answer</summary>

Set `return_source_documents=True` when creating the chain:  
```python
qa = RetrievalQA.from_chain_type(
    ...,
    return_source_documents=True
)
result = qa.invoke({"query": "..."})
sources = result['source_documents']  # List of Document objects
```
</details>

---

## ✅ Module 7 Complete!

**Key Takeaways:**
1. Full RAG pipeline: Load → Chunk → Embed → Store → Retrieve → Generate
2. Chain types: **stuff** (default), map_reduce (large docs), refine (iterative), map_rerank (best single)
3. LCEL (`|` pipes) is the modern approach; RetrievalQA is legacy but still tested
4. `RunnablePassthrough()` passes input unchanged through the chain
5. `format_docs()` helper converts Document objects to a context string
6. Source attribution with `return_source_documents=True` or `RunnableParallel`

**Next:** [Module 08 — Prompting for RAG](./08_Prompting_for_RAG.ipynb)